# 11 — CRAG: Corrective RAG Correction Layer

Implements the Phase-4 piece of RAG-Lab's original architecture that was scoped early on but never built: a correction layer sitting between retrieval and generation, from Yan et al.'s *Corrective Retrieval Augmented Generation* (2024).

The core idea: don't trust retrieval blindly. Grade what came back, and route the query down one of three paths depending on the verdict:

| Path | When | What happens |
|---|---|---|
| **Correct** | every retrieved doc is relevant | refine them (strip to the sentences that actually matter) and generate |
| **Incorrect** | every retrieved doc is irrelevant | discard them entirely, substitute a web search |
| **Ambiguous** | mixed verdict, or any doc graded ambiguous | refine the relevant internal docs **and** web search, combine both |

This is implemented as `CRAGStrategy` in `rag_lab/strategies/crag.py`, with the same `.run(query) -> str` interface as every other strategy in this project, plus `.run_with_trace(query) -> dict` for inspecting *why* it took the path it took — which is most of the point of a demo notebook for a correction layer.

**Two deliberate simplifications** (documented in detail in the module docstring, repeated briefly here since they affect what you'll see below):
- Knowledge refinement (the paper's decompose → grade-each-strip →   recompose) collapses into **one** structured-output call per correct   document that extracts the relevant strips directly — same result,   fewer calls against whatever's left of the day's token budget.
- The web-search query rewrite is a single plain LLM call, only on the   Incorrect/Ambiguous paths — not on every query.

## Setup

Same corpus, same API-key pattern as every other notebook here. Grading is **1 LLM call per retrieved document** (plus up to 1 more per correct document for refinement, plus 1 for a web-search query rewrite when triggered) — meaningfully more calls per question than naive/multi_query, so this notebook splits the load across two models: a cheap/local `grader_llm` for grading + refinement + rewrite, and whichever `llm` you've been using for final-answer generation. Flip `USE_LOCAL_OLLAMA` to run grading locally (recommended) while keeping generation on Groq, or set both to the same model if you'd rather not run two providers.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import getpass
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········


In [2]:
%pip install -q ddgs

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
dense_vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory= "~/Codes/rag-lab/data/chroma_naive_gemini",
)
print(f"{dense_vectorstore._collection.count()} chunks loaded")

576 chunks loaded


In [13]:
USE_LOCAL_OLLAMA_FOR_GRADING = False  # keeps the per-document grading calls off Groq's TPD

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
#llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)  
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

if USE_LOCAL_OLLAMA_FOR_GRADING:
    from langchain_ollama import ChatOllama
    OLLAMA_BASE_URL = "http://localhost:11434" 
    grader_llm = ChatOllama(model="qwen2.5:7b", base_url=OLLAMA_BASE_URL, temperature=0)
else:
    grader_llm = llm

print(f"generation llm: {llm}")
print(f"grader llm: {grader_llm}")

generation llm: metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.17', 'langchain-google-genai': '4.4.0'}} profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'} google_api_key=SecretStr('**********') model='gemini-3.6-flash' temperature=None client=<google.genai.client.Client object at 0x7f9d58d4b490> default_metadata=() model_kwargs={}
grader llm: metadata={

In [15]:
from rag_lab.strategies.crag import CRAGStrategy

crag = CRAGStrategy(
    dense_vectorstore,
    llm=llm,
    grader_llm=grader_llm,
    k=4,
    verbose=True,
)

## Trace pretty-printer

In [9]:
def show_trace(trace):
    print(f"Query: {trace['query']}")
    print(f"Path taken: {trace['path'].upper()}")
    print()
    print("Grades:")
    for g in trace["grades"]:
        print(f"  [{g['grade']:>9}] {g['preview']!r}...")
        print(f"             reason: {g['reason']}")
    if trace["web_search_query"]:
        print()
        print(f"Web search query: {trace['web_search_query']!r}")
        for r in trace["web_results"]:
            print(f"  - {r['preview']!r}... ({r['metadata'].get('url') or r['metadata'].get('source')})")
    print()
    print("Answer:")
    print(trace["answer"])

## Demo 1 — expect the CORRECT path

A query squarely inside the corpus, same test query used in `01_foundation.ipynb` and `08_colbert.ipynb` — the dense retriever has already been shown to return genuinely relevant chunks for this one.

In [10]:
trace_correct = crag.run_with_trace(
    "What is a Physics Informed Neural Network and how is it different from a Graphical Neural Network?"
)
show_trace(trace_correct)

[crag] graded 4 docs: ['ambiguous', 'ambiguous', 'ambiguous', 'ambiguous']
[crag] refined 3/4 docs into knowledge strips
[crag] web search ('Physics Informed Neural Network vs Graph Neural Network differences') -> 4 results
Query: What is a Physics Informed Neural Network and how is it different from a Graphical Neural Network?
Path taken: AMBIGUOUS

Grades:
  [ambiguous] 'partial differential equations, Science Advances 3 (2017) e1602614. URL: https:\n//www.science.org/doi/abs/10.1126/sciadv'...
             reason: The document lists references about physics-informed neural networks and a physics-informed graph neural Galerkin network, touching both topics, but provides no explanatory content to define PINNs or contrast them with Graph Neural Networks.
  [ambiguous] 'PIGNN 3\nIn addition to physics-based models, data-driven methods [22, 23, 24] also seek\nnonlinear mappings from parametr'...
             reason: The document discusses a physics‑informed graph neural network (PIGNN) a

## Demo 2 — expect the AMBIGUOUS path

A query that touches the corpus's subject area (PDE-solving neural methods) but asks about something the 11-paper corpus likely doesn't cover in depth — a good candidate for a mixed or ambiguous grading verdict, where some retrieved chunks are topically close without actually answering the question.

In [11]:
trace_ambiguous = crag.run_with_trace(
    "How does a Fourier Neural Operator compare to a Physics-Informed Neural Network for solving PDEs?"
)
show_trace(trace_ambiguous)

[crag] graded 4 docs: ['ambiguous', 'incorrect', 'incorrect', 'ambiguous']
[crag] refined 0/2 docs into knowledge strips
[crag] web search ('Fourier Neural Operator vs Physics-Informed Neural Network PDE solving comparison') -> 4 results
Query: How does a Fourier Neural Operator compare to a Physics-Informed Neural Network for solving PDEs?
Path taken: AMBIGUOUS

Grades:
  [ambiguous] 'partial differential equations, Science Advances 3 (2017) e1602614. URL: https:\n//www.science.org/doi/abs/10.1126/sciadv'...
             reason: The document lists references about Physics-Informed Neural Networks but contains no information about Fourier Neural Operators, so it only partially addresses the comparison.
  [incorrect] '2 \n \n \n1. Introduction \nPartial differential equations (PDEs) provide a concise and effective framework for \naddressing '...
             reason: The document discusses PDEs and PINNs but does not mention Fourier Neural Operators or any comparison between them, so it 

## Demo 3 — expect the INCORRECT path

A query with nothing to do with the corpus at all, to force every retrieved doc to grade as irrelevant and trigger the pure web-search fallback. If this sandbox/network can't reach DuckDuckGo, `default_web_search` degrades to a placeholder document rather than crashing — check `trace_incorrect['web_results']` to see which happened.

In [16]:
trace_incorrect = crag.run_with_trace(
    "What is the current exchange rate between the US dollar and the Japanese yen?"
)
show_trace(trace_incorrect)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[crag] graded 4 docs: ['incorrect', 'incorrect', 'incorrect', 'incorrect']
[crag] web search ('USD to JPY exchange rate today') -> 4 results
Query: What is the current exchange rate between the US dollar and the Japanese yen?
Path taken: INCORRECT

Grades:
  [incorrect] 'A PREPRINT - JULY 30, 2020\nwhere\nI1 = 1\nN\nN∑\nk=1\n(\nW (1)\nk\n)2(\nW (0)\nk\n)4...σ(W (0)\nk x +b(0)\nk )x· ...σ(W (0)\nk x′ +b(0'...
             reason: The document contains mathematical equations from a preprint and has no relevance to currency exchange rates.
  [incorrect] 'dθ ,du(xj\nb,θ(t))\ndθ\n⟩\n.\nSimilarly,\ndLu(xj\nr,θ(t))\ndt = dLu(xj\nr,θ(t))\ndθ\nT\n· dθ\ndt\n= dLu(xj\nr,θ(t))\ndθ\nT\n·\n[ Nb∑\ni=1\n(u'...
             reason: The document contains mathematical formulas and differential equations that are completely unrelated to currency exchange rates.
  [incorrect] 'By lemma D.1, it is easy to show that‖J(t)‖2 is bounded. So it now sufﬁces to show that\nsup\nt∈[0,T]\n‖Ju(t)−Ju(0)‖F→ 0 (D'...
   

## Where this fits in the rest of RAG-Lab

- **Interface-compatible with the eval harness.** `CRAGStrategy(vectorstore,   llm=llm).run(query)` matches every other strategy class, and   `STRATEGY_CALL_COUNTS["crag"]` is already added to `rag_lab/utils.py`   (pacing) — adding `"crag": CRAGStrategy` to the `STRATEGIES` dict in   `09_colbert_generation.ipynb` is enough to fold it into the same   checkpointed generation+scoring pipeline as the other six, across both   backends, if/when that comparison is worth running. Not done in this   notebook — grading cost (1+ extra LLM call per retrieved doc, on top of   generation) makes a full 63-question x 2-backend CRAG run meaningfully   more expensive than the other six strategies, worth a deliberate   decision rather than folding in by default.
- **Next pieces still open from the original architecture**: the   adaptive query router (dispatch to best-performing strategy by query   type, informed by whatever the eval numbers end up showing), and the   final README/write-up.